# 03 · Autoencoders for Anomaly Detection

## What this notebook covers
Autoencoders learn to compress and reconstruct normal data. Anomalies, by definition underrepresented in training, reconstruct poorly — giving a natural anomaly score: reconstruction error.

## The core idea
Train an encoder-decoder to minimise reconstruction loss on normal data only. At inference time, high reconstruction error = anomaly. The bottleneck forces the network to learn a compressed representation of normality; anything that doesn't fit that representation won't reconstruct well.

## Architecture choices
- **Bottleneck size**: too small → poor reconstruction of normal data; too large → anomalies also reconstruct well (the network memorises everything)
- **Loss function**: MSE is standard for continuous features; BCE for binary/bounded features
- **Threshold**: the reconstruction error threshold determines the operating point. Set it from a held-out normal validation set at a target FPR.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42); torch.manual_seed(42)

# Dataset: normal = two tight clusters; anomalies = uniform noise
X_normal, _ = make_blobs(n_samples=1000, centers=2, cluster_std=0.5, random_state=0, n_features=16)
X_anom = np.random.uniform(-6, 6, (50, 16))
X_all = np.vstack([X_normal, X_anom])
y_all = np.array([0]*1000 + [1]*50)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_normal)
X_test_np  = scaler.transform(X_all)

X_train = torch.FloatTensor(X_train_np)
X_test  = torch.FloatTensor(X_test_np)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape} ({y_all.sum()} anomalies)")

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, bottleneck)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 16), nn.ReLU(),
            nn.Linear(16, 32),         nn.ReLU(),
            nn.Linear(32, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def reconstruction_error(self, x):
        with torch.no_grad():
            recon = self.forward(x)
            return ((x - recon) ** 2).mean(dim=1).numpy()

model = Autoencoder(input_dim=16, bottleneck=4)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train), batch_size=64, shuffle=True)

losses = []
for epoch in range(80):
    epoch_loss = 0
    for (batch,) in loader:
        recon = model(batch)
        loss = nn.MSELoss()(recon, batch)
        optimiser.zero_grad(); loss.backward(); optimiser.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))
print(f"Final train loss: {losses[-1]:.6f}")

In [ ]:
# Score test set
recon_errors = model.reconstruction_error(X_test)
auc  = roc_auc_score(y_all, recon_errors)
ap   = average_precision_score(y_all, recon_errors)
print(f"Autoencoder: ROC-AUC={auc:.4f}  AP={ap:.4f}")

# Threshold from validation: 95th percentile of normal reconstruction errors
train_errors = model.reconstruction_error(X_train)
threshold = np.percentile(train_errors, 95)
preds = recon_errors > threshold
prec = (preds & y_all.astype(bool)).sum() / preds.sum()
rec  = (preds & y_all.astype(bool)).sum() / y_all.sum()
print(f"Threshold (95th pct of train errors): {threshold:.4f}")
print(f"  Flagged: {preds.sum()}  Precision: {prec:.3f}  Recall: {rec:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(losses, color="steelblue")
axes[0].set_title("Training loss"); axes[0].set_xlabel("Epoch")

axes[1].hist(recon_errors[y_all==0], bins=40, alpha=0.7, color="steelblue", label="Normal", density=True)
axes[1].hist(recon_errors[y_all==1], bins=20, alpha=0.7, color="tomato",    label="Anomaly", density=True)
axes[1].axvline(threshold, color="black", linestyle="--", label=f"Threshold={threshold:.3f}")
axes[1].set_title("Reconstruction error distribution")
axes[1].set_xlabel("MSE"); axes[1].legend(fontsize=8)

# Bottleneck size ablation
bottlenecks = [2, 4, 8, 16]
auc_by_bn = []
for bn in bottlenecks:
    m = Autoencoder(16, bn)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(60):
        for (b,) in DataLoader(TensorDataset(X_train), batch_size=64, shuffle=True):
            l = nn.MSELoss()(m(b), b)
            opt.zero_grad(); l.backward(); opt.step()
    errs = m.reconstruction_error(X_test)
    auc_by_bn.append(roc_auc_score(y_all, errs))

axes[2].plot(bottlenecks, auc_by_bn, "o-", color="steelblue")
axes[2].set_title("AUC vs bottleneck size"); axes[2].set_xlabel("Bottleneck dim"); axes[2].set_ylabel("ROC-AUC")

plt.tight_layout()
plt.savefig(f"{base}/03_autoencoder.png", dpi=120)
plt.show()